In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
#1. pdf loader

from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf").load()

for d in pages:
    d.metadata["page_no"] = d.metadata["page"] + 1

print(f"페이지 {len(pages)}개")


페이지 30개


In [10]:
#2. splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(pages)

In [11]:
#3. 임베딩

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(len(embeddings.embed_query("테스트")))

1536


In [12]:
#4. pinecone 연결
import os
import time
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_KEY"))

index_name = "rag-test"   # 소문자 + 하이픈

if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# 인덱스가 준비될 때까지 대기
while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

# 인덱스 연결
index = pc.Index(index_name)

d:\yujin0902\rag_two_branch\rag_test2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
#5. 벡터 스토어 pinecone, cosine

from langchain_pinecone import PineconeVectorStore

index = pc.Index(index_name)

vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

vectorstore.add_documents(
    documents=chunks,
    ids=[f"chunk-{i}" for i in range(len(chunks))],   # 고정 ID: 재실행해도 중복 저장 안 됨
)

# 업로드 직후에는 반영이 늦을 수 있어 개수가 맞을 때까지 대기
for _ in range(30):
    count = index.describe_index_stats()["total_vector_count"]
    if count >= len(chunks):
        break
    time.sleep(1)

print("split 청크 수:", len(chunks))
print("저장된 벡터 수:", count)

split 청크 수: 83
저장된 벡터 수: 83


In [ ]:
# #ChromaDB 저장
# from langchain_chroma import Chroma

# DB_PATH = "./chroma_db"
# COLLECTION_NAME = "ai_ethics_guide"

# vectorstore = Chroma.from_documents(
#     documents=split_docs,
#     embedding=embeddings,
#     collection_name=COLLECTION_NAME,
#     persist_directory=DB_PATH,
# )


In [14]:
#6. 리트리버

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    },
)
test_docs = retriever.invoke("가짜 뉴스를 만들어 배포하면 어떤 경우에 법적 책임이 생기나요?")
print("검색된 청크 수:", len(test_docs))
print("검색된 페이지:", [int(d.metadata["page_no"]) for d in test_docs])

검색된 청크 수: 5
검색된 페이지: [16, 17, 17, 10, 7]


In [15]:
#7. 유사도

def search_with_scores(query, k=5):
    """질문과 코사인 유사도가 높은 순서대로 청크를 출력하고 반환한다."""
    hits = vectorstore.similarity_search_with_score(query, k=k)
    for rank, (doc, similarity) in enumerate(hits, start=1):
        page = int(doc.metadata["page_no"])   # Pinecone은 숫자를 float로 저장함
        preview = doc.page_content[:80].replace("\n", " ")
        print(f"{rank}위 | 유사도 {similarity:.4f} | p.{page} | {preview}")
    return hits


query = "가짜 뉴스를 만들어 배포하면 어떤 경우에 법적 책임이 생기나요?"
search_with_scores(query)

1위 | 유사도 0.5573 | p.16 | 43                                                                              
2위 | 유사도 0.5208 | p.17 | 피해를 줄 목적이 없었다고 할지라도 해당 뉴스로 인해 직접적인 손해를 입은 사람이 존재한다면 이러한  손해에 대해서는 민사상 손해배상 책임을 부
3위 | 유사도 0.4753 | p.17 | 44 관련사례 ▶  AI발 가짜 뉴스에 눈 뜨고 코 베이는데…규제는 미비(연합뉴스TV 2023. 7. 25.) 최근 한 일본 기자가 프랑스 축구
4위 | 유사도 0.4053 | p.10 | 사용자들의 책임의식 미정립 탓에 발생한 헤프닝이라고 할 수 있습니다.   AI가 다 했으니 사람은 책임 없다? 디지털 콘텐츠를 제작하거나 활용할
5위 | 유사도 0.4017 | p.7 | 지위는 원칙적으로 권리능력이 전제가 되어야 한다. (중략) 인공지능은 법령상 자연인과 법인  모두에 포섭되지 않으므로 현행 법령상으로 인공지능에


[(Document(id='chunk-42', metadata={'author': '방송미디어통신위원회·한국지능정보사회진흥원', 'creationdate': '', 'creator': 'PyPDF', 'keywords': '생성형 AI, 윤리, RAG, 시험용 발췌', 'page': 15.0, 'page_label': '16', 'page_no': 16.0, 'producer': 'pypdf', 'source': 'data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf', 'sourcepages': '1,10,13,14,18,19,20,27,28,32,33,34,35,36,44,45,46,50,51,54,55,56,57,62,63,67,68,69,71,73', 'subject': 'RAG 프로젝트 시험용 핵심 발췌본', 'title': '생성형 AI윤리 가이드북 - RAG 시험용 30쪽 발췌본', 'total_pages': 30.0}, page_content='43\n  \n   \n             \n               \n             \n               \n               \n       \n                 \n             \n  \n   \n              \n            \n              \n      \n                \n             \n             \n               \n      \n1    재미로 생성형 AI를 이용해서 가짜 뉴스를 만들어 배포했어요. 누군가에게 \n피해를 줄 목적이 아니었는데도 처벌받나요? \n세상에 존재하지 않는 내용을 다루는 가짜 정보를 기반으로 한 가짜 뉴스를 만들어 배포하는 것 자체는 \n위법하다고 볼 수는 없습니다. 그러나 누군가에게 피해를 줄 목적이 없었더라도, 이러한 가짜 뉴스가 \n실존하는 개인 혹은 법인의 명예를 훼손하는 내용을 다루고 있거나, 이러한 내용으로 인

In [17]:
#8. prompt

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
        """당신은 '생성형 AI윤리 가이드북'에 대해 답하는 도우미입니다.
다음 문서를 참고해서 질문에 답변하세요.

[참고 문서]
{context}

[질문]
{question}

문서에 있는 내용만 사용해서 답변하세요.
문서에서 답을 찾을 수 없다면, "문서에서 확인할 수 없습니다."라고 답변하세요.

[답변]"""
)



In [18]:
#9. llm

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0,
    model_name="gpt-4o-mini"
)

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(retrieved_docs):
    return "\n\n".join(d.page_content for d in retrieved_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
